In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle

import structlog
import logging
structlog.configure(
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
)

import sys
sys.path.append('../../../../')

import pickle

from src.difsched.agents.dr3rlpy import train_bc, evaluate, train_iql, train_td3bc, train_cql
from src.difsched.agents.gym_env import HybridEnv
from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\Ye\miniconda3\envs\traffic_predictor_3_9\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
trafficDatasetFolder = f'../../../../data/processed/traffic'
dataset_path = "offline_dataset.pkl"
with open(dataset_path, "rb") as f:
    dataset = pickle.load(f)

expConfigIdx = 1
expParams = getExpConfig(expConfigIdx)
visualizeExpConfig(expParams)

simEnv = createEnv(expParams, trafficDatasetFolder)
simEnv.selectMode(mode="test", type="data")

EnvType: HYBRID
N_user: 20
LEN_window: 20
N_aggregation: 4
dataflow: thumb_fr
randomSeed: 999
r_bar: 4
B: 100
sigma_list: [0.7, 0.75, 0.8, 0.85, 0.9]
offline_dataset_idxs: [6, 7, 8]


In [3]:
evaluate_ep = 5

bc = train_bc(dataset, device="cuda", n_steps=5000, n_steps_per_epoch=100)
iql = train_iql(dataset, device="cuda", n_steps=5000, n_steps_per_epoch=100)
cql = train_cql(dataset, device="cuda", n_steps=5000, n_steps_per_epoch=100)

#td3bc = train_td3bc(dataset, device="cuda", n_steps=5000, n_steps_per_epoch=100)

Epoch 50/50: 100%|██████████| 100/100 [00:09<00:00, 10.42it/s, critic_loss=402, conservative_loss=-837, alpha=0.581, actor_loss=594, temp=1.43, temp_loss=-42.1]


In [4]:
env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=5000)
packet_loss_bc = 1 - evaluate(bc, env, n_episodes=evaluate_ep)
print(f"BC avg packet loss rate: {packet_loss_bc:.6f}")

env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=5000)
packet_loss_iql = 1 - evaluate(iql, env, n_episodes=evaluate_ep)
print(f"IQL avg packet loss rate: {packet_loss_iql:.6f}")

env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=5000)
packet_loss_cql = 1 - evaluate(cql, env, n_episodes=evaluate_ep)
print(f"CQL avg packet loss rate: {packet_loss_cql:.6f}")

# env = HybridEnv(expParams, simEnv, obvMode="predicted", max_episode_steps=5000)
# packet_loss_td3bc = 1 - evaluate(td3bc, env, n_episodes=evaluate_ep)
# print(f"TD3+BC avg packet loss rate: {packet_loss_td3bc:.6f}")

BC avg packet loss rate: 0.076405
IQL avg packet loss rate: 0.071470
CQL avg packet loss rate: 0.077064
